In [1]:
import os
os.getcwd()

'C:\\Users\\TWH'

In [2]:
import pandas as pd
import numpy as np

# Automatically install and import mlxtend if missing
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    %pip install mlxtend
    from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
# ==============================================================================
# 1. LOAD DATASET
# ==============================================================================
df = pd.read_csv("data/processed/cleaned_traffic_features.csv")

In [4]:
# ==============================================================================
# 2. CATEGORICAL DISCRETIZATION (Feature Engineering for Market Basket)
# ==============================================================================
def get_time_of_day(hour):
    if 6 <= hour < 10:
        return 'Time: Morning Rush'
    elif 10 <= hour < 16:
        return 'Time: Midday'
    elif 16 <= hour < 20:
        return 'Time: Evening Rush'
    elif 20 <= hour < 23:
        return 'Time: Late Evening'
    else:
        return 'Time: Night'

df['time_category'] = df['hour'].apply(get_time_of_day)
df['day_type'] = np.where(df['is_weekend'] == 1, 'DayType: Weekend', 'DayType: Weekday')

def get_weather_category(row):
    if row['is_severe_weather'] == 1:
        return 'Weather: Severe'
    elif row['is_low_visibility'] == 1:
        return 'Weather: Low Visibility'
    else:
        return 'Weather: Normal'

df['weather_category'] = df.apply(get_weather_category, axis=1)
df['congestion_category_item'] = 'Congestion: ' + df['congestion_category']

In [5]:
# ==============================================================================
# 3. BUILD TRANSACTION MATRIX (basket_df)
# ==============================================================================
basket_df = pd.get_dummies(
    df[['time_category', 'day_type', 'weather_category', 'congestion_category_item']],
    prefix='',
    prefix_sep=''
).astype(bool)

In [6]:
# ==============================================================================
# 4. MINE FREQUENT ITEMSETS & GENERATE RULES
# ==============================================================================
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.1)

# Filter rules where the consequent predicts a Congestion status
congestion_rules = rules[
    rules['consequents'].apply(lambda x: any('Congestion:' in item for item in x))
].sort_values(by='lift', ascending=False)

In [7]:
# ==============================================================================
# 5. DISPLAY TOP RULES
# ==============================================================================
top_rules = congestion_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5)

print("=== TOP 5 ASSOCIATION RULES BY LIFT ===")
for idx, row in top_rules.reset_index(drop=True).iterrows():
    ant = list(row['antecedents'])
    con = list(row['consequents'])
    print(f"Rule {idx+1}: {ant}  ==>  {con}")
    print(f"  • Support: {row['support']:.4f} | Confidence: {row['confidence']:.4f} | Lift: {row['lift']:.4f}\n")

=== TOP 5 ASSOCIATION RULES BY LIFT ===
Rule 1: ['Time: Morning Rush', 'DayType: Weekday']  ==>  ['Weather: Low Visibility', 'Congestion: Severe']
  • Support: 0.0250 | Confidence: 0.2045 | Lift: 5.0351

Rule 2: ['Time: Late Evening']  ==>  ['Weather: Normal', 'DayType: Weekday', 'Congestion: Medium']
  • Support: 0.0664 | Confidence: 0.5372 | Lift: 4.6036

Rule 3: ['Time: Late Evening']  ==>  ['DayType: Weekday', 'Congestion: Medium']
  • Support: 0.0828 | Confidence: 0.6697 | Lift: 4.3427

Rule 4: ['Time: Late Evening', 'Weather: Normal']  ==>  ['DayType: Weekday', 'Congestion: Medium']
  • Support: 0.0664 | Confidence: 0.6680 | Lift: 4.3316

Rule 5: ['Time: Late Evening', 'DayType: Weekday']  ==>  ['Weather: Normal', 'Congestion: Medium']
  • Support: 0.0664 | Confidence: 0.7492 | Lift: 4.0076

